# Explore Study Graph

This notebook shows how to move from a study lookup to richer notebook UI:

1. Connect and sample a few studies
2. Pick one study alias
3. Inspect its versions in a table
4. Query a path-shaped neighborhood
5. Render that path as a graph


Prerequisite:

```bash
aws sso login --profile dsoadev
```


In [ ]:
import {
  connectNotebook,
  sampleStudies,
  studyNeighborhood,
  studyVersions,
  ui
} from "@sdr-notebook/mod";

const session = await connectNotebook({ profile: "dsoadev" });
const studies = await sampleStudies(session, { minVersions: 4, limit: 5 });

ui.table(studies);


## Pick a Study

Use one of the sampled aliases from the previous cell, or keep the fallback if the sample is empty.


In [2]:
const studyAlias = studies[0]?.alias ?? "CRD-09-2036";

ui.md`## Exploring\n\nStudy alias: **${studyAlias}**`;


ReferenceError: studies is not defined

## Versions Table

This uses the `studyVersions(...)` helper, which already queries by study alias correctly.


In [4]:
const versions = await studyVersions(session, studyAlias);

ui.table(
  versions.map((version) => ({
    id: version.id,
    name: version.name,
    instanceType: version.instanceType,
  })),
);


ReferenceError: studyVersions is not defined

## Graph Render

For graph visualization, query `.path()` results and then render them explicitly with `ui.graph(...)`.

### What `studyNeighborhood(...)` does

`studyNeighborhood(session, alias, { hops, limit })` is a small helper around a Gremlin path query. It returns the raw `.path()` result as an array of path objects.

The helper currently runs this query pattern:

```gremlin
g.V().has("name", "<study alias>").hasLabel("Study")
  .repeat(outE().inV().simplePath()).times(<hops>)
  .path().limit(<limit>)
```

If you want a different neighborhood shape, you can skip the helper and pass your own `.path()` query to `session.gremlin(...)`.


In [ ]:
const neighborhood = await studyNeighborhood(session, studyAlias, { hops: 3, limit: 20 });

ui.graph(neighborhood, {
  width: 1500,
  height: 900,
  linkDistance: 190,
  chargeStrength: -500,
  nodeRadius: 32,
  nodeFontSize: 10,
  edgeFontSize: 9,
  labelMaxChars: 22,
  showNodeType: true,
  showEdgeLabels: true,
});
